# 03_Modelado y Clasificación

**Objetivo**: Construir un modelo de clasificación que, a partir de las características extraídas en el notebook anterior, sea capaz de diagnosticar el tipo de accidente nuclear entre las 18 clases disponibles.

**Estrategia**:
1. Usar un **Random Forest** para seleccionar las características más relevantes (las 64 primeras acumulan ~89% de la importancia total).
2. Entrenar un **nuevo Random Forest** con esas 64 características.
3. Evaluar el modelo en test y mediante validación cruzada.

**Justificación del modelo**: Random Forest es un algoritmo robusto, interpretable (permite extraer importancia de características) y no requiere escalado de los datos, lo que lo hace ideal para este tipo de problemas tabulares.

---

## 1. Configuración inicial

Importamos las librerías necesarias, configuramos el proyecto y definimos los parámetros del modelo.

In [1]:
import sys
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Añadir la raíz del proyecto al path para importar src
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.features import seleccionar_columnas_por_importancia
from src.model import entrenar_modelo, evaluar_modelo, validacion_cruzada, guardar_modelo
from src.utils import crear_carpeta, cargar_dataset

# ===== PARÁMETROS =====
DATA_PATH = "../data/processed/features_12sensores_1217muestras.csv"
MODEL_PATH = "../results/modelos/modelo_rf.joblib"
RANDOM_STATE = 42          # Semilla para reproducibilidad
TEST_SIZE = 0.2            # 20% de los datos para test
UMBRAL_IMPORTANCIA = 0.007 # Selecciona características con importancia > 0.007
N_ESTIMATORS = 100         # Número de árboles en el Random Forest

## 2. Carga de datos y división train/test
Cargamos el dataset generado en el notebook anterior (`features_12sensores_1217muestras.csv`). Contiene 1217 muestras y 110 características (108 estadísticas + `accidente` + `severidad`).

La variable objetivo es el tipo de accidente (`accidente`), que codificamos con `LabelEncoder`. Dividimos los datos en entrenamiento (80%) y prueba (20%) manteniendo la misma semilla para reproducibilidad.

In [2]:
print("Cargando dataset...")
X, y, le = cargar_dataset(DATA_PATH)
print(f"Dataset cargado: {X.shape[0]} muestras, {X.shape[1]} características")
print(f"Clases: {le.classes_}")

# División estratificada (mantiene proporción de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]} muestras, Test: {X_test.shape[0]} muestras")

Cargando dataset...
Dataset cargado: 1217 muestras, 109 características
Clases: ['ATWS' 'FLB' 'LACP' 'LLB' 'LOCA' 'LOCAC' 'LOF' 'LR' 'MD' 'Normal' 'RI'
 'RW' 'SGATR' 'SGBTR' 'SLBIC' 'SLBOC' 'SP' 'TT']
Train: 973 muestras, Test: 244 muestras


## 3. Selección de características por importancia
Entrenamos un Random Forest temporal para obtener la importancia de cada una de las 109 características (excluyendo a la variable objetivo `accidente`). Seleccionamos aquellas cuya importancia supera el umbral de 0.007. Este umbral se eligió empíricamente para quedarnos con las 64 características que acumulan aproximadamente el 89% de la importancia total, reduciendo la dimensionalidad sin perder capacidad predictiva.

In [3]:
print("\nSeleccionando características por importancia...")
columnas_seleccionadas = seleccionar_columnas_por_importancia(
    X_train, y_train,
    umbral=UMBRAL_IMPORTANCIA,
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_STATE
)

# Filtrar los conjuntos de entrenamiento y prueba
X_train_filt = X_train[columnas_seleccionadas]
X_test_filt = X_test[columnas_seleccionadas]


Seleccionando características por importancia...
Seleccionadas 61 características con umbral 0.007


## 4. Entrenamiento del modelo final
Entrenamos un nuevo Random Forest con las 64 características seleccionadas. Usamos los mismos hiperparámetros que en el modelo temporal para mantener coherencia.

In [4]:
print("\nEntrenando modelo final...")
model = entrenar_modelo(
    X_train_filt, y_train,
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_STATE
)
print("¡Modelo entrenado!")


Entrenando modelo final...
¡Modelo entrenado!


## 5. Evaluación del modelo
Evaluamos el modelo en el conjunto de test y mediante validación cruzada estratificada con 5 folds.

**Métricas**: Accuracy y classification report (precisión, recall, F1-score por clase).

In [5]:
print("\n--- Evaluación en test ---")
y_pred, acc = evaluar_modelo(model, X_test_filt, y_test, target_names=le.classes_)

print("\n--- Validación cruzada (5 folds) ---")
cv_mean, cv_std = validacion_cruzada(model, X_train_filt, y_train, cv=5, scoring='accuracy')


--- Evaluación en test ---
Accuracy en test: 0.9713

Classification Report (clases en test):
              precision    recall  f1-score   support

         FLB       1.00      1.00      1.00        17
         LLB       1.00      1.00      1.00        21
        LOCA       1.00      1.00      1.00        19
       LOCAC       1.00      1.00      1.00        19
          LR       0.96      1.00      0.98        24
          MD       0.92      0.86      0.89        14
      Normal       0.00      0.00      0.00         1
          RI       0.92      1.00      0.96        24
          RW       1.00      0.91      0.95        22
       SGATR       1.00      1.00      1.00        22
       SGBTR       1.00      1.00      1.00        25
       SLBIC       1.00      1.00      1.00        22
       SLBOC       0.86      0.86      0.86        14

   micro avg       0.98      0.97      0.97       244
   macro avg       0.90      0.89      0.90       244
weighted avg       0.97      0.97      0

## 6. Guardado del modelo y columnas seleccionadas
Guardamos el modelo entrenado y la lista de columnas seleccionadas para poder reutilizarlos en el siguiente notebook (evaluación e interpretación con SHAP).

In [6]:
# Crear carpeta si no existe
crear_carpeta(os.path.dirname(MODEL_PATH))

# Guardar modelo
guardar_modelo(model, MODEL_PATH)

# Guardar columnas seleccionadas
columnas_path = "../results/columnas_seleccionadas.txt"
np.savetxt(columnas_path, columnas_seleccionadas, fmt="%s")
print(f"Columnas seleccionadas guardadas en {columnas_path}")

Modelo guardado en ../results/modelos/modelo_rf.joblib
Columnas seleccionadas guardadas en ../results/columnas_seleccionadas.txt


## 7. Conclusiones
* Hemos reducido el número de características de 110 a **64**, manteniendo el 89% de la importancia total.
* El modelo final alcanza una **accuracy en test > 97%**, lo que indica un excelente rendimiento.
* La validación cruzada confirma la estabilidad del modelo (media ≈ 0.98, desviación < 0.01).
* El siguiente paso será **interpretar el modelo** con SHAP para entender qué características son más influyentes y cómo toma decisiones en casos concretos.

**Nota**: La alta precisión obtenida es prometedora, pero en un entorno real sería necesario validar el modelo con datos de planta (no simulados) y considerar la calibración de probabilidades.